# Video to 3D model (Colab, free GPU)

Run the cells from top to bottom. **Before you start:** menu *Runtime > Change runtime type > T4 GPU*.

1. Cell 1 checks the GPU. 2. Cell 2 installs COLMAP (about 2 minutes). 3. Cell 3 asks for your video.
4. Cells 4 to 6 find the cameras and build the dense point cloud (about 20 to 60 minutes).
5. Cell 7 **keeps only the piece** (drops the wall and floor) and shows a before/after picture.
6. Cell 8 turns the piece upright, sets its **real size**, makes the surface and a **front-view depth map** for carving.
7. Cell 9 downloads the files.

**Filming for a flat or half-round carving (most pieces):** walk a slow arc about 60 degrees to each side of straight-on, at two or three heights.
Fill the frame with the piece. Put a **patterned matte cloth** behind it (a plain white wall gives the software nothing to match).
Put a **ruler or marked stick** next to it. Even light, no zooming, move slowly so frames are not blurred.

**For a full 3D object:** walk all the way around, plus passes from above and below.

In [ ]:
# 1. GPU check
import subprocess
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(out.stdout or out.stderr)
if "GPU" not in out.stdout:
    raise SystemExit("No GPU. Use Runtime > Change runtime type > T4 GPU, then run again.")

In [ ]:
# 2. Install COLMAP with CUDA (pip wheel) and helpers
!pip -q install pycolmap-cuda12 trimesh pillow matplotlib
import pycolmap
print("pycolmap", pycolmap.__version__, "| CUDA build:", getattr(pycolmap, "has_cuda", "unknown"))
if getattr(pycolmap, "has_cuda", True) is False:
    raise SystemExit("This pycolmap has no CUDA, so the dense step cannot run. Paste this message to Claude.")

In [ ]:
# 3. Upload the video (a pop-up asks for the file)
from google.colab import files
from pathlib import Path
import shutil

work = Path("/content/work")
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)
up = files.upload()
video = work / Path(next(iter(up))).name
Path(next(iter(up))).rename(video)
print("video:", video, round(video.stat().st_size / 1e6, 1), "MB")

In [ ]:
# 4. Pull still pictures out of the video (about 120 frames, longest side 1600 px)
import subprocess, shutil
TARGET_FRAMES = 120
MAX_SIDE = 1600

dur = float(subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(video)],
    capture_output=True, text=True, check=True).stdout.strip())
fps = max(0.5, min(4.0, TARGET_FRAMES / dur))
images = work / "images"
shutil.rmtree(images, ignore_errors=True)
images.mkdir()
subprocess.run([
    "ffmpeg", "-loglevel", "error", "-i", str(video),
    "-vf", f"fps={fps:.3f},scale='if(gt(iw,ih),{MAX_SIDE},-2)':'if(gt(iw,ih),-2,{MAX_SIDE})'",
    "-q:v", "2", str(images / "f%04d.jpg")], check=True)
frames = sorted(images.glob("*.jpg"))
print(f"{dur:.0f} s of video -> {len(frames)} pictures at {fps:.2f} per second")

In [ ]:
# 5. Find where the camera was for every picture (sparse model). 5 to 15 minutes.
import pycolmap, time
t0 = time.time()
db = work / "database.db"
sparse = work / "sparse"
sparse.mkdir(exist_ok=True)
if db.exists():
    db.unlink()

pycolmap.extract_features(db, images, camera_mode=pycolmap.CameraMode.SINGLE)
if len(frames) <= 150:
    pycolmap.match_exhaustive(db)
else:
    pycolmap.match_sequential(db)
maps = pycolmap.incremental_mapping(db, images, sparse)
if not maps:
    raise SystemExit("COLMAP could not place the cameras. Try a slower video with more overlap.")
best = max(maps, key=lambda k: maps[k].num_reg_images())
rec = maps[best]
print(rec.summary())
print(f"placed {rec.num_reg_images()} of {len(frames)} pictures in {time.time() - t0:.0f} s")
if rec.num_reg_images() < 0.5 * len(frames):
    print("WARNING: fewer than half the pictures were placed, the model may be partial.")

In [ ]:
# 6. Dense depth for every picture and fuse into a point cloud (GPU). 10 to 40 minutes.
t0 = time.time()
mvs = work / "mvs"
pycolmap.undistort_images(mvs, sparse / str(best), images)
opts = pycolmap.PatchMatchOptions()
opts.max_image_size = 1600
opts.cache_size = 8
pycolmap.patch_match_stereo(mvs, options=opts)
pycolmap.stereo_fusion(mvs / "fused.ply", mvs, output_type="ply")
print(f"dense cloud done in {time.time() - t0:.0f} s")

In [ ]:
# 7. Keep only the piece: the wall, floor and stray points are dropped. Look at the before/after picture.
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

# Settings. Leave them as they are the first time; change only if the piece is cut off or the wall is still there.
DENSITY_PERCENTILE = None   # None = choose by itself. Higher (for example 90) keeps less; lower (for example 60) keeps more.
CROP_BOX = None             # or ((xmin, ymin, zmin), (xmax, ymax, zmax)) in COLMAP units, to set the box yourself

PLY_TYPES = {"float": "<f4", "uchar": "u1"}


def read_ply(path):
    """COLMAP's fused.ply (binary): a numpy array with x y z nx ny nz red green blue."""
    with open(path, "rb") as f:
        lines = []
        while not lines or lines[-1] != "end_header":
            lines.append(f.readline().decode("ascii").strip())
        if "binary_little_endian" not in lines[1]:
            raise SystemExit("This PLY file is not binary little-endian. Paste this message to Claude.")
        n = int(next(l for l in lines if l.startswith("element vertex")).split()[-1])
        dt = np.dtype([(l.split()[2], PLY_TYPES[l.split()[1]]) for l in lines if l.startswith("property")])
        return np.frombuffer(f.read(n * dt.itemsize), dtype=dt, count=n)


def write_ply(path, v):
    head = ["ply", "format binary_little_endian 1.0", f"element vertex {len(v)}"]
    head += [f"property {'float' if v.dtype[n].kind == 'f' else 'uchar'} {n}" for n in v.dtype.names] + ["end_header"]
    with open(path, "wb") as f:
        f.write(("\n".join(head) + "\n").encode("ascii"))
        f.write(v.tobytes())


def find_piece(xyz, percentile=None):
    """Which points are the piece: the biggest solid blob of densely packed points (the wall and floor are thinner and sparser)."""
    lo, hi = np.percentile(xyz, [1, 99], axis=0)
    scene = float(np.prod(hi - lo))
    idx = np.floor((xyz - xyz.min(0)) / (float((hi - lo).max()) / 150)).astype(np.int64)
    count = np.zeros(idx.max(0) + 1, np.int32)
    np.add.at(count, tuple(idx.T), 1)
    occupied = count[count > 0]

    def blob(q):
        dense = count >= np.percentile(occupied, q)
        labels, n = ndimage.label(dense, structure=np.ones((3, 3, 3)))
        if n == 0:
            return None
        biggest = 1 + int(np.argmax(ndimage.sum(dense, labels, range(1, n + 1))))
        solid = ndimage.binary_dilation(labels == biggest, iterations=1)  # take the skin around it too
        return solid[tuple(idx.T)]

    if percentile:
        return blob(percentile), percentile
    found = {q: blob(q) for q in range(50, 100, 5)}
    volume = {q: float(np.prod(np.ptp(xyz[k], axis=0))) if k is not None and k.any() else np.inf for q, k in found.items()}
    for q in range(50, 95, 5):
        # the piece: small next to the whole scene, and its size has stopped shrinking as the setting rises (the wall is gone)
        if volume[q] < 0.10 * scene and volume[q] <= 1.25 * volume[q + 5]:
            return found[q], q
    return found[50], 50  # no small, stable piece found: keep the densest half; set DENSITY_PERCENTILE or CROP_BOX yourself


cloud = read_ply(mvs / "fused.ply")
xyz = np.column_stack([cloud["x"], cloud["y"], cloud["z"]]).astype(np.float64)
if CROP_BOX is None:
    keep, q = find_piece(xyz, DENSITY_PERCENTILE)
else:
    lo, hi = (np.array(b, float) for b in CROP_BOX)
    keep, q = ((xyz >= lo) & (xyz <= hi)).all(1), "box"
piece = cloud[keep]
pts = xyz[keep]
print(f"kept {len(piece):,} of {len(cloud):,} points ({100 * len(piece) / len(cloud):.0f}%), setting {q}")
print("piece size in COLMAP units (x, y, z):", np.round(np.ptp(pts, axis=0), 2), "| whole scene:", np.round(np.ptp(xyz, axis=0), 1))

rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, (P, C, title) in enumerate([(xyz, cloud, "everything"), (pts, piece, "kept")]):
    pick = rng.choice(len(P), min(50000, len(P)), replace=False)
    col = np.column_stack([C["red"], C["green"], C["blue"]])[pick] / 255
    for ax, (a, b, name) in zip(axes[row], [(0, 1, "x-y"), (0, 2, "x-z"), (1, 2, "y-z")]):
        ax.scatter(P[pick, a], P[pick, b], s=0.3, c=col)
        ax.set_aspect("equal")
        ax.set_title(f"{title} ({name})")
plt.show()

In [ ]:
# 8. Stand the piece upright, set its real size, make the surface (GLB/STL) and the front-view depth map for carving.
import trimesh
from PIL import Image

REAL_HEIGHT_MM = None   # the real height in mm of the piece as seen from the front (for example 900). Needed for real sizes.


def camera_poses(rec):
    """Where each camera was, and which way was up for it (the video was filmed upright)."""
    centres, ups = [], []
    for im in rec.images.values():
        if not getattr(im, "has_pose", True):
            continue
        pose = im.cam_from_world
        pose = pose() if callable(pose) else pose
        centres.append(np.asarray(im.projection_center(), float))
        ups.append(-np.asarray(pose.rotation.matrix(), float)[1, :])  # COLMAP's camera y points down
    return np.array(centres), np.array(ups)


def front_frame(points, centres=None, ups=None):
    """(right, up, toward-the-viewer) as a right-handed frame, so the result can never be mirrored."""
    mid = points.mean(0)
    w, v = np.linalg.eigh(np.cov((points - mid).T))  # v[:, 0] = thinnest direction, v[:, 2] = longest
    if centres is not None and len(centres):
        towards = centres.mean(0) - mid
        if np.linalg.norm(towards) < 0.25 * np.linalg.norm(centres - mid, axis=1).mean():
            print("note: the cameras went all the way round, so there is no single front. Using the piece's flattest side.")
            towards = v[:, 0] * (1 if ((centres - mid) @ v[:, 0]).sum() >= 0 else -1)
    else:
        print("note: camera positions were not available. Guessed the front from the piece's shape; check the picture.")
        towards = v[:, 0]
    D = towards / np.linalg.norm(towards)
    U = ups.mean(0) if ups is not None and len(ups) else v[:, 2]
    U = U - (U @ D) * D
    if np.linalg.norm(U) < 1e-3:
        U = v[:, 2] - (v[:, 2] @ D) * D
    U /= np.linalg.norm(U)
    return np.cross(U, D), U, D


def depth_map(points, frame, long_side_px=700):
    """Front-view height map: 16-bit, white = nearest. Plus a shaded picture to look at.
    The size comes from the cleaned silhouette (thin fringes of stray points are trimmed), not from the outermost points."""
    R, U, D = frame
    x, y, z = (points - points.mean(0)) @ R, (points - points.mean(0)) @ U, (points - points.mean(0)) @ D
    px = max(np.ptp(x), np.ptp(y)) / long_side_px
    w, h = int(np.ptp(x) / px) + 1, int(np.ptp(y) / px) + 1
    ix, iy = ((x - x.min()) / px).astype(int), ((y.max() - y) / px).astype(int)
    depth = np.full((h, w), -np.inf)
    np.maximum.at(depth, (iy, ix), z)
    seen = np.isfinite(depth)
    mask = ndimage.binary_closing(ndimage.binary_dilation(seen, iterations=1), iterations=3)
    mask = ndimage.binary_opening(mask, iterations=3)  # trim thin fringes
    labels, n = ndimage.label(mask)
    if n > 1:
        mask = labels == (1 + int(np.argmax(ndimage.sum(mask, labels, range(1, n + 1)))))  # keep the one solid shape
    mask = ndimage.binary_fill_holes(mask)
    nearest = ndimage.distance_transform_edt(~seen, return_distances=False, return_indices=True)
    filled = ndimage.median_filter(np.where(seen, depth, depth[tuple(nearest)]), size=3)
    # stray points stick up as spikes and a cutter would carve every one: flatten what is far from its neighbourhood, then smooth a little
    around = ndimage.median_filter(filled, size=9)
    off = filled - around
    filled = np.where(np.abs(off) > 4 * (np.median(np.abs(off[mask])) + 1e-9), around, filled)
    filled = ndimage.gaussian_filter(filled, 1.0)
    low, high = np.percentile(filled[mask], [1, 99.5])
    n_ = np.clip((filled - low) / (high - low), 0, 1)
    n_[~mask] = 0
    rows, cols = np.where(mask.any(1))[0], np.where(mask.any(0))[0]
    r0, r1, c0, c1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1  # crop to the figure
    n_, mask = n_[r0:r1, c0:c1], mask[r0:r1, c0:c1]
    gy, gx = np.gradient(ndimage.gaussian_filter(n_, 1.0) * 40)
    shade = np.clip(0.5 + 0.5 * (-gx * 0.6 - gy * 0.6) / np.sqrt(1 + gx**2 + gy**2), 0, 1)
    shade[~mask] = 0.08
    return (n_ * 65535).astype(np.uint16), (shade * 255).astype(np.uint8), (mask.shape[1] * px, mask.shape[0] * px, float(high - low))


try:
    centres, ups = camera_poses(rec)
except Exception as err:  # a different pycolmap version: fall back to the piece's shape
    print("could not read camera poses:", type(err).__name__, err)
    centres = ups = None
frame = front_frame(pts, centres, ups)
depth16, shade8, (width_u, height_u, relief_u) = depth_map(pts, frame)
Image.fromarray(depth16).save(work / "figure_depth.png")
Image.fromarray(shade8).save(work / "figure_preview.png")

# surface of the piece only
write_ply(work / "piece.ply", piece)
try:
    opts = pycolmap.PoissonMeshingOptions()
    opts.depth = 11  # fewer and cleaner triangles than the default
    pycolmap.poisson_meshing(work / "piece.ply", work / "meshed.ply", options=opts)
except (AttributeError, TypeError):  # a pycolmap that words these options differently: use the defaults
    pycolmap.poisson_meshing(work / "piece.ply", work / "meshed.ply")
mesh = max(trimesh.load(work / "meshed.ply", force="mesh").split(only_watertight=False), key=lambda m: len(m.faces))
mesh.vertices = (np.asarray(mesh.vertices) - pts.mean(0)) @ np.column_stack(frame)  # x right, y up, z toward the viewer

print(f"piece: {width_u:.2f} wide x {height_u:.2f} tall x {relief_u:.2f} deep (COLMAP units) | mesh faces: {len(mesh.faces):,}")
if REAL_HEIGHT_MM:
    k = REAL_HEIGHT_MM / height_u
    mesh.apply_scale(k)
    print(f"real size: {width_u * k:.0f} mm wide x {REAL_HEIGHT_MM:.0f} mm tall x {relief_u * k:.0f} mm deep")
    print("carving command (run on the workshop PC):")
    print(f"  python vriksha.py figure_depth.png --width {width_u * k:.0f} --max-depth {relief_u * k:.0f} --floor 0.05 -o figure.nc --preview figure_cut.png")
    print("  (lower --max-depth if your wood block is thinner; deeper reliefs are squeezed to fit)")
else:
    print("not scaled: set REAL_HEIGHT_MM above to get real sizes and the carving command.")
mesh.export(work / "model.stl")  # millimetres when scaled
glb = mesh.copy()
if REAL_HEIGHT_MM:
    glb.apply_scale(0.001)  # glTF is in metres
glb.export(work / "model.glb")

fig, axes = plt.subplots(1, 2, figsize=(10, 7))
axes[0].imshow(shade8, cmap="gray")
axes[0].set_title("front view, shaded (should be upright and not mirrored)")
axes[1].imshow(depth16, cmap="viridis")
axes[1].set_title("depth: bright = nearest")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
# 9. Download the results
from google.colab import files
for name in ["figure_depth.png", "figure_preview.png", "model.glb", "model.stl", "piece.ply"]:
    files.download(str(work / name))

**What you get:** `figure_depth.png` (16-bit, white = nearest; the input for `vriksha.py`), `figure_preview.png` (look at it first),
`model.glb` and `model.stl` (the piece only, upright, in real size if you set `REAL_HEIGHT_MM`), `piece.ply` (the cleaned coloured points).

**If the front view is upside down or the wall is still there:** change the settings at the top of cell 7 or 8 and run that cell again.